In [4]:
import xarray as xr
import cfgrib
import airportsdata
import pandas as pd
#import numpy as np
#import os, sys
from datetime import date, timedelta

# Provides latitude, longitude information for airports based on IATA code
iata_airports = airportsdata.load('IATA')

In [ ]:
# Loads the paths used for the weather data, flight data, and the folder where combined data will be delivered

path_grib = '/Users/joaopedrocarvalho/Documents/erdos_spring_2026/weather-data/'
path_flights = '/Users/joaopedrocarvalho/Documents/erdos_spring_2026/airline-data/'
path_dump = '/Users/joaopedrocarvalho/Documents/erdos_spring_2026/combined-data/'

In [5]:
path_grib = r'C:\Users\David\Documents\GitHub\spring-2026-airline-reliability\data\\'
path_flights = r'C:\Users\David\Documents\GitHub\spring-2026-airline-reliability\data\Flight_data\\'
path_dump =r'C:\Users\David\Documents\GitHub\spring-2026-airline-reliability\combined-data\\'


In [12]:
years = [2023]

In [ ]:
# Transforms GRIB weather files into NetCDF format. 
# Each of our grib files has 2 component datasets - one with 5 variables and one with 2, due to different heights, so we get 2 NetCDF files per GRIB
# This is done as GRIB is the source format and more memory efficient for storage, but NetCDF speeds up the combining process by about 4x

for year in years:
    if year == 2020:
        weather = cfgrib.open_datasets(path_grib + '2020_dec_weather.grib', engine = 'cfgrib')
        weather[0].to_netcdf(path_grib + 'weather_2020_dec_step.nc')
        weather[1].to_netcdf(path_grib + 'weather_2020_dec_uv.nc')
    elif year == 2025:
        weather = cfgrib.open_datasets(path_grib + '2025_jan_to_nov_weather.grib', engine = 'cfgrib')
        weather[0].to_netcdf(path_grib + 'weather_2025_jan_to_nov_step.nc')
        weather[1].to_netcdf(path_grib + 'weather_2025_jan_to_nov_uv.nc')
    else:
        weather = cfgrib.open_datasets(path_grib + f'{year}_jan_to_dec_weather.grib', engine = 'cfgrib')
        weather[0].to_netcdf(path_grib + f'weather_{year}_jan_to_dec_step.nc')
        weather[1].to_netcdf(path_grib + f'weather_{year}_jan_to_dec_uv.nc')

c:\Users\David\.conda\envs\erdos_ds_environment\Lib\site-packages\cfgrib\xarray_store.py:51: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  o = xr.merge([o, ds], **kwargs)
c:\Users\David\.conda\envs\erdos_ds_environment\Lib\site-packages\cfgrib\xarray_store.py:51: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  o = xr.merge([o, ds], **kw

In [5]:
# Takes year, month, day, and hour (as numbers) and spits out strings encoding that information in the weather data format
# One file uses "baseline" times and 12h steps from there, the other uses 24h format, this returns all three

def hour_step(year, month, day, time):
    baseline = ''
    whole_hour = 0
    hour_int = 0
    step = 0
    hour_int = 0
    today = date(year, month, day)

    # Transforms the time integer into the approximate hour of day in 24h format
    if (time/100) % 1 < 0.3:
        hour_int = int(time/100)
    else: hour_int = int(time/100) + 1

    # Transforms date and approximate hour into one weather file format, making sure we always stay in the dataset currently open
    # Conditions are specific to the timeframe we are using the data over, and hoe it is partitioned (Dec 2020 + Jan-Dec 20{21, 22, 23, 24} + Jan-Nov 2025)
    if hour_int < 10:
        whole_hour = str(today) + 'T0' + str(hour_int) + ':00:00'
    elif hour_int == 24 :
        if today.year == 2025 and today.month == (today + timedelta(days=1)).month:
            whole_hour = str(today + timedelta(days=1)) + 'T00:00:00'       
        elif today.year != 2025 and today.year == (today + timedelta(days=1)).year:
            whole_hour = str(today + timedelta(days=1)) + 'T00:00:00'
        else: whole_hour = str(today) + 'T23:00:00'
    else: whole_hour = str(today) + 'T' + str(hour_int) + ':00:00'

    # Transforms date and hour into the format with baselines at 6am and 18pm + step out from there
    if 6 < hour_int and hour_int <= 18:
        baseline = 'T06:00:00'
        if hour_int - 6 < 10:
            step = '0' + str(hour_int - 6) + ':00:00'
        else: step = str(hour_int - 6) + ':00:00'
    elif hour_int > 18:
        baseline = 'T18:00:00'
        step = '0' + str(hour_int - 18) + ':00:00'
    elif hour_int <= 6:
        baseline = 'T18:00:00'
        if hour_int + 6 < 10:
            step = '0' + str(hour_int + 6) + ':00:00'
        else: step = str(hour_int + 6) + ':00:00'
        today = today - timedelta(days=1)
    hour = str(today) + baseline 
    
    return [hour, step, whole_hour]

In [8]:
###The lag for getting the correct format for the time 2 hours before the input.
def hour_step_2(year, month, day, time):
    baseline = ''
    whole_hour = 0
    hour_int = 0
    step = 0
    hour_int = 0
    today = date(year, month, day)

    # Transforms the time integer into the approximate hour of day in 24h format
    if (time/100) % 1 < 0.3:
        hour_int = int(time/100)
    else: hour_int = int(time/100) + 1

    if hour_int>2:
        hour_int = hour_int - 2
    else: 
        hour_int = hour_int +22
        today = today - timedelta(days=1)

    # Transforms date and approximate hour into one weather file format, making sure we always stay in the dataset currently open
    # Conditions are specific to the timeframe we are using the data over, and hoe it is partitioned (Dec 2020 + Jan-Dec 20{21, 22, 23, 24} + Jan-Nov 2025)
    if hour_int < 10:
        whole_hour = str(today) + 'T0' + str(hour_int) + ':00:00'
    elif hour_int == 24 :
        if today.year == 2025 and today.month == (today + timedelta(days=1)).month:
            whole_hour = str(today + timedelta(days=1)) + 'T00:00:00'       
        elif today.year != 2025 and today.year == (today + timedelta(days=1)).year:
            whole_hour = str(today + timedelta(days=1)) + 'T00:00:00'
        else: whole_hour = str(today) + 'T23:00:00'
    else: whole_hour = str(today) + 'T' + str(hour_int) + ':00:00'

    # Transforms date and hour into the format with baselines at 6am and 18pm + step out from there
    if 6 < hour_int and hour_int <= 18:
        baseline = 'T06:00:00'
        if hour_int - 6 < 10:
            step = '0' + str(hour_int - 6) + ':00:00'
        else: step = str(hour_int - 6) + ':00:00'
    elif hour_int > 18:
        baseline = 'T18:00:00'
        step = '0' + str(hour_int - 18) + ':00:00'
    elif hour_int <= 6:
        baseline = 'T18:00:00'
        if hour_int + 6 < 10:
            step = '0' + str(hour_int + 6) + ':00:00'
        else: step = str(hour_int + 6) + ':00:00'
        today = today - timedelta(days=1)
    hour = str(today) + baseline 
    
    return [hour, step, whole_hour]

In [9]:
###The lag for getting the correct format for the time 4 hours before the input.
def hour_step_4(year, month, day, time):
    baseline = ''
    whole_hour = 0
    hour_int = 0
    step = 0
    hour_int = 0
    today = date(year, month, day)

    # Transforms the time integer into the approximate hour of day in 24h format
    if (time/100) % 1 < 0.3:
        hour_int = int(time/100)
    else: hour_int = int(time/100) + 1

    if hour_int>4:
        hour_int = hour_int - 4
    else: 
        hour_int = hour_int +20
        today = today - timedelta(days=1)

    # Transforms date and approximate hour into one weather file format, making sure we always stay in the dataset currently open
    # Conditions are specific to the timeframe we are using the data over, and hoe it is partitioned (Dec 2020 + Jan-Dec 20{21, 22, 23, 24} + Jan-Nov 2025)
    if hour_int < 10:
        whole_hour = str(today) + 'T0' + str(hour_int) + ':00:00'
    elif hour_int == 24 :
        if today.year == 2025 and today.month == (today + timedelta(days=1)).month:
            whole_hour = str(today + timedelta(days=1)) + 'T00:00:00'       
        elif today.year != 2025 and today.year == (today + timedelta(days=1)).year:
            whole_hour = str(today + timedelta(days=1)) + 'T00:00:00'
        else: whole_hour = str(today) + 'T23:00:00'
    else: whole_hour = str(today) + 'T' + str(hour_int) + ':00:00'

    # Transforms date and hour into the format with baselines at 6am and 18pm + step out from there
    if 6 < hour_int and hour_int <= 18:
        baseline = 'T06:00:00'
        if hour_int - 6 < 10:
            step = '0' + str(hour_int - 6) + ':00:00'
        else: step = str(hour_int - 6) + ':00:00'
    elif hour_int > 18:
        baseline = 'T18:00:00'
        step = '0' + str(hour_int - 18) + ':00:00'
    elif hour_int <= 6:
        baseline = 'T18:00:00'
        if hour_int + 6 < 10:
            step = '0' + str(hour_int + 6) + ':00:00'
        else: step = str(hour_int + 6) + ':00:00'
        today = today - timedelta(days=1)
    hour = str(today) + baseline 
    
    return [hour, step, whole_hour]

In [129]:
# Takes in IATA airport, returns longitude and latitude approximated to nearest 1/4 degree (coarseness of weather data)

def lon_lat(airport):
    lon_raw = iata_airports[airport]['lon']
    lat_raw = iata_airports[airport]['lat']
    lon = round(lon_raw * 4)/4
    lat = round(lat_raw * 4)/4
    return [lon, lat]

In [130]:
# Takes in date, time, location, and the 2 split NetCDF weather files, returns temperature, precipitation, and wind information

def weather_data(year, month, day, time, lon, lat, alt_weather_1, alt_weather_2):
    hour, step, whole_hour = hour_step(year, month, day, time)
    #lon, lat = lon_lat(airport)
    mx2t = alt_weather_1['mx2t'].sel(time = hour, step = step, latitude = lat, longitude = lon).item()
    mn2t = alt_weather_1['mn2t'].sel(time = hour, step = step, latitude = lat, longitude = lon).item()
    tp = alt_weather_1['tp'].sel(time = hour, step = step, latitude = lat, longitude = lon).item()
    mxtpr = alt_weather_1['mxtpr'].sel(time = hour, step = step, latitude = lat, longitude = lon).item()
    fg10 = alt_weather_1['fg10'].sel(time = hour, step = step, latitude = lat, longitude = lon).item()
    u10 = alt_weather_2['u10'].sel(time = whole_hour, latitude = lat, longitude = lon).item()
    v10 = alt_weather_2['v10'].sel(time = whole_hour, latitude = lat, longitude = lon).item()
    return [mx2t, mn2t, tp, mxtpr, fg10, u10, v10]

In [10]:
## Analogous function for lags
def weather_data_2(year, month, day, time, lon, lat, alt_weather_1, alt_weather_2):
    hour, step, whole_hour = hour_step_2(year, month, day, time)
    #lon, lat = lon_lat(airport)
    mx2t = alt_weather_1['mx2t'].sel(time = hour, step = step, latitude = lat, longitude = lon).item()
    mn2t = alt_weather_1['mn2t'].sel(time = hour, step = step, latitude = lat, longitude = lon).item()
    tp = alt_weather_1['tp'].sel(time = hour, step = step, latitude = lat, longitude = lon).item()
    mxtpr = alt_weather_1['mxtpr'].sel(time = hour, step = step, latitude = lat, longitude = lon).item()
    fg10 = alt_weather_1['fg10'].sel(time = hour, step = step, latitude = lat, longitude = lon).item()
    u10 = alt_weather_2['u10'].sel(time = whole_hour, latitude = lat, longitude = lon).item()
    v10 = alt_weather_2['v10'].sel(time = whole_hour, latitude = lat, longitude = lon).item()
    return [mx2t, mn2t, tp, mxtpr, fg10, u10, v10]


def weather_data_4(year, month, day, time, lon, lat, alt_weather_1, alt_weather_2):
    hour, step, whole_hour = hour_step_4(year, month, day, time)
    #lon, lat = lon_lat(airport)
    mx2t = alt_weather_1['mx2t'].sel(time = hour, step = step, latitude = lat, longitude = lon).item()
    mn2t = alt_weather_1['mn2t'].sel(time = hour, step = step, latitude = lat, longitude = lon).item()
    tp = alt_weather_1['tp'].sel(time = hour, step = step, latitude = lat, longitude = lon).item()
    mxtpr = alt_weather_1['mxtpr'].sel(time = hour, step = step, latitude = lat, longitude = lon).item()
    fg10 = alt_weather_1['fg10'].sel(time = hour, step = step, latitude = lat, longitude = lon).item()
    u10 = alt_weather_2['u10'].sel(time = whole_hour, latitude = lat, longitude = lon).item()
    v10 = alt_weather_2['v10'].sel(time = whole_hour, latitude = lat, longitude = lon).item()
    return [mx2t, mn2t, tp, mxtpr, fg10, u10, v10]




In [115]:
# Takes in the pre-cleaned flight data, does some extra processing, and returns a new CSV file with added columns for departure and arrival weather information
# It takes on average 2.7s per 100 flights, this process is very time-intensive - can likely benefit greatly from parallelization

for year in years:

    # Conditions are based on the months of each year we are using, each month is then treated in the same way
    if year == 2020:

        # Opens flight and weather files, adds departure and arrival longitute and latitude data to flights
        # Excludes any leftover rows missing information from previous pre-processing and any flights whose origin or destination is not in the continental USA
        flights = pd.read_csv(path_flights + 'cleaned_2020_12.csv')
        alt_weather_1 = xr.open_dataset(path_grib + 'weather_2020_dec_step.nc')
        alt_weather_2 = xr.open_dataset(path_grib + 'weather_2020_dec_uv.nc')
        flights[['dep_longitude', 'dep_latitude']] = flights.apply(lambda x: lon_lat(x['Origin']), axis = 1, result_type = 'expand')
        flights[['arr_longitude', 'arr_latitude']] = flights.apply(lambda x: lon_lat(x['Dest']), axis = 1, result_type = 'expand')
        flights = flights.dropna()
        in_window_dep = (-125 <= flights['dep_longitude']) & (flights['dep_longitude'] <= -66) & (24 <= flights['dep_latitude']) & (flights['dep_latitude'] <= 50)
        flights = flights[in_window_dep]
        in_window_arr = (-125 <= flights['arr_longitude']) & (flights['arr_longitude'] <= -66) & (24 <= flights['arr_latitude']) & (flights['arr_latitude'] <= 50)
        flights = flights[in_window_arr]

        # Adds weather data to flights file, convert appropriate column types to less memory-intensive options - 
        flights[['dep_mx2t', 'dep_mn2t', 'dep_tp', 'dep_mxtpr', 'dep_fg10', 'dep_u10', 'dep_v10']] = flights.apply(
            lambda x: weather_data(x['Year'], x['Month'], x['DayofMonth'], x['CRSDepTime'], x['dep_longitude'], x['dep_latitude'], alt_weather_1, alt_weather_2), 
            axis = 1, result_type = 'expand')
        flights[['arr_mx2t', 'arr_mn2t', 'arr_tp', 'arr_mxtpr', 'arr_fg10', 'arr_u10', 'arr_v10']] = flights.apply(
            lambda x: weather_data(x['Year'], x['Month'], x['DayofMonth'], x['CRSDepTime'], x['arr_longitude'], x['arr_latitude'], alt_weather_1, alt_weather_2), 
            axis = 1, result_type = 'expand')
        for column in flights.columns:
            if flights[column].dtype == 'int64':
                flights[column] = flights[column].astype('int32')
            if flights[column].dtype == 'float64':
                flights[column] = flights[column].astype('float32')
        flights.to_csv(path_dump + '2020_12_with_weather.csv')
        

    elif year == 2025:
        alt_weather_1 = xr.open_dataset(path_grib + 'weather_2025_jan_to_nov_step.nc')
        alt_weather_2 = xr.open_dataset(path_grib + 'weather_2025_jan_to_nov_uv.nc')
        for month in range(1,12):
            flights = pd.read_csv(path_flights + f'cleaned_2025_{month}.csv')
            flights[['dep_longitude', 'dep_latitude']] = flights.apply(lambda x: lon_lat(x['Origin']), axis = 1, result_type = 'expand')
            flights[['arr_longitude', 'arr_latitude']] = flights.apply(lambda x: lon_lat(x['Dest']), axis = 1, result_type = 'expand')
            flights = flights.dropna()
            in_window_dep = (-125 <= flights['dep_longitude']) & (flights['dep_longitude'] <= -66) & (24 <= flights['dep_latitude']) & (flights['dep_latitude'] <= 50)
            flights = flights[in_window_dep]
            in_window_arr = (-125 <= flights['arr_longitude']) & (flights['arr_longitude'] <= -66) & (24 <= flights['arr_latitude']) & (flights['arr_latitude'] <= 50)
            flights = flights[in_window_arr]
            flights[['dep_mx2t', 'dep_mn2t', 'dep_tp', 'dep_mxtpr', 'dep_fg10', 'dep_u10', 'dep_v10']] = flights.apply(
                lambda x: weather_data(x['Year'], x['Month'], x['DayofMonth'], x['CRSDepTime'], x['dep_longitude'], x['dep_latitude'], alt_weather_1, alt_weather_2), 
                axis = 1, result_type = 'expand')
            flights[['arr_mx2t', 'arr_mn2t', 'arr_tp', 'arr_mxtpr', 'arr_fg10', 'arr_u10', 'arr_v10']] = flights.apply(
                lambda x: weather_data(x['Year'], x['Month'], x['DayofMonth'], x['CRSDepTime'], x['arr_longitude'], x['arr_latitude'], alt_weather_1, alt_weather_2), 
                axis = 1, result_type = 'expand')
            for column in flights.columns:
                if flights[column].dtype == 'int64':
                    flights[column] = flights[column].astype('int32')
                if flights[column].dtype == 'float64':
                    flights[column] = flights[column].astype('float32')
            flights.to_csv(path_dump + f'2025_{month}_with_weather.csv')
                
            

    else: 
        alt_weather_1 = xr.open_dataset(path_grib + f'weather_{year}_jan_to_dec_step.nc')
        alt_weather_2 = xr.open_dataset(path_grib + f'weather_{year}_jan_to_dec_uv.nc')
        for month in range(1,13):
            flights = pd.read_csv(path_flights + f'cleaned_{year}_{month}.csv')
            flights[['dep_longitude', 'dep_latitude']] = flights.apply(lambda x: lon_lat(x['Origin']), axis = 1, result_type = 'expand')
            flights[['arr_longitude', 'arr_latitude']] = flights.apply(lambda x: lon_lat(x['Dest']), axis = 1, result_type = 'expand')
            flights = flights.dropna()
            in_window_dep = (-125 <= flights['dep_longitude']) & (flights['dep_longitude'] <= -66) & (24 <= flights['dep_latitude']) & (flights['dep_latitude'] <= 50)
            flights = flights[in_window_dep]
            in_window_arr = (-125 <= flights['arr_longitude']) & (flights['arr_longitude'] <= -66) & (24 <= flights['arr_latitude']) & (flights['arr_latitude'] <= 50)
            flights = flights[in_window_arr]
            flights[['dep_mx2t', 'dep_mn2t', 'dep_tp', 'dep_mxtpr', 'dep_fg10', 'dep_u10', 'dep_v10']] = flights.apply(
                lambda x: weather_data(x['Year'], x['Month'], x['DayofMonth'], x['CRSDepTime'], x['dep_longitude'], x['dep_latitude'], alt_weather_1, alt_weather_2), 
                axis = 1, result_type = 'expand')
            flights[['arr_mx2t', 'arr_mn2t', 'arr_tp', 'arr_mxtpr', 'arr_fg10', 'arr_u10', 'arr_v10']] = flights.apply(
                lambda x: weather_data(x['Year'], x['Month'], x['DayofMonth'], x['CRSDepTime'], x['arr_longitude'], x['arr_latitude'], alt_weather_1, alt_weather_2), 
                axis = 1, result_type = 'expand')
            for column in flights.columns:
                if flights[column].dtype == 'int64':
                    flights[column] = flights[column].astype('int32')
                if flights[column].dtype == 'float64':
                    flights[column] = flights[column].astype('float32')
            flights.to_csv(path_dump + f'{year}_{month}_with_weather.csv')
        

In [ ]:
##Added time lag of 2 and 4 to the above code, fixed the CSRArr for the else loop
##If running time is too slow, run the next cell instead, where only a time lag of 2 is added

for year in years:

    # Conditions are based on the months of each year we are using, each month is then treated in the same way
    if year == 2020:

        # Opens flight and weather files, adds departure and arrival longitute and latitude data to flights
        # Excludes any leftover rows missing information from previous pre-processing and any flights whose origin or destination is not in the continental USA
        flights = pd.read_csv(path_flights + 'cleaned_2020_12.csv')
        alt_weather_1 = xr.open_dataset(path_grib + 'weather_2020_dec_step.nc')
        alt_weather_2 = xr.open_dataset(path_grib + 'weather_2020_dec_uv.nc')
        flights[['dep_longitude', 'dep_latitude']] = flights.apply(lambda x: lon_lat(x['Origin']), axis = 1, result_type = 'expand')
        flights[['arr_longitude', 'arr_latitude']] = flights.apply(lambda x: lon_lat(x['Dest']), axis = 1, result_type = 'expand')
        flights = flights.dropna()
        in_window_dep = (-125 <= flights['dep_longitude']) & (flights['dep_longitude'] <= -66) & (24 <= flights['dep_latitude']) & (flights['dep_latitude'] <= 50)
        flights = flights[in_window_dep]
        in_window_arr = (-125 <= flights['arr_longitude']) & (flights['arr_longitude'] <= -66) & (24 <= flights['arr_latitude']) & (flights['arr_latitude'] <= 50)
        flights = flights[in_window_arr]

        # Adds weather data to flights file, convert appropriate column types to less memory-intensive options - 
        flights[['dep_mx2t', 'dep_mn2t', 'dep_tp', 'dep_mxtpr', 'dep_fg10', 'dep_u10', 'dep_v10']] = flights.apply(
            lambda x: weather_data(x['Year'], x['Month'], x['DayofMonth'], x['CRSDepTime'], x['dep_longitude'], x['dep_latitude'], alt_weather_1, alt_weather_2), 
            axis = 1, result_type = 'expand')
        flights[['arr_mx2t', 'arr_mn2t', 'arr_tp', 'arr_mxtpr', 'arr_fg10', 'arr_u10', 'arr_v10']] = flights.apply(
            lambda x: weather_data(x['Year'], x['Month'], x['DayofMonth'], x['CRSDepTime'], x['arr_longitude'], x['arr_latitude'], alt_weather_1, alt_weather_2), 
            axis = 1, result_type = 'expand')
        for column in flights.columns:
            if flights[column].dtype == 'int64':
                flights[column] = flights[column].astype('int32')
            if flights[column].dtype == 'float64':
                flights[column] = flights[column].astype('float32')
        flights.to_csv(path_dump + '2020_12_with_weather.csv')
        

    elif year == 2025:
        alt_weather_1 = xr.open_dataset(path_grib + 'weather_2025_jan_to_nov_step.nc')
        alt_weather_2 = xr.open_dataset(path_grib + 'weather_2025_jan_to_nov_uv.nc')
        for month in range(1,12):
            flights = pd.read_csv(path_flights + f'cleaned_2025_{month}.csv')
            flights[['dep_longitude', 'dep_latitude']] = flights.apply(lambda x: lon_lat(x['Origin']), axis = 1, result_type = 'expand')
            flights[['arr_longitude', 'arr_latitude']] = flights.apply(lambda x: lon_lat(x['Dest']), axis = 1, result_type = 'expand')
            flights = flights.dropna()
            in_window_dep = (-125 <= flights['dep_longitude']) & (flights['dep_longitude'] <= -66) & (24 <= flights['dep_latitude']) & (flights['dep_latitude'] <= 50)
            flights = flights[in_window_dep]
            in_window_arr = (-125 <= flights['arr_longitude']) & (flights['arr_longitude'] <= -66) & (24 <= flights['arr_latitude']) & (flights['arr_latitude'] <= 50)
            flights = flights[in_window_arr]
            flights[['dep_mx2t', 'dep_mn2t', 'dep_tp', 'dep_mxtpr', 'dep_fg10', 'dep_u10', 'dep_v10']] = flights.apply(
                lambda x: weather_data(x['Year'], x['Month'], x['DayofMonth'], x['CRSDepTime'], x['dep_longitude'], x['dep_latitude'], alt_weather_1, alt_weather_2), 
                axis = 1, result_type = 'expand')
            flights[['arr_mx2t', 'arr_mn2t', 'arr_tp', 'arr_mxtpr', 'arr_fg10', 'arr_u10', 'arr_v10']] = flights.apply(
                lambda x: weather_data(x['Year'], x['Month'], x['DayofMonth'], x['CRSDepTime'], x['arr_longitude'], x['arr_latitude'], alt_weather_1, alt_weather_2), 
                axis = 1, result_type = 'expand')
            for column in flights.columns:
                if flights[column].dtype == 'int64':
                    flights[column] = flights[column].astype('int32')
                if flights[column].dtype == 'float64':
                    flights[column] = flights[column].astype('float32')
            flights.to_csv(path_dump + f'2025_{month}_with_weather.csv')
                
            

    else: 
        alt_weather_1 = xr.open_dataset(path_grib + f'weather_{year}_jan_to_dec_step.nc')
        alt_weather_2 = xr.open_dataset(path_grib + f'weather_{year}_jan_to_dec_uv.nc')
        for month in range(1,13):
            flights = pd.read_csv(path_flights + f'cleaned_{year}_{month}.csv')
            flights[['dep_longitude', 'dep_latitude']] = flights.apply(lambda x: lon_lat(x['Origin']), axis = 1, result_type = 'expand')
            flights[['arr_longitude', 'arr_latitude']] = flights.apply(lambda x: lon_lat(x['Dest']), axis = 1, result_type = 'expand')
            flights = flights.dropna()
            in_window_dep = (-125 <= flights['dep_longitude']) & (flights['dep_longitude'] <= -66) & (24 <= flights['dep_latitude']) & (flights['dep_latitude'] <= 50)
            flights = flights[in_window_dep]
            in_window_arr = (-125 <= flights['arr_longitude']) & (flights['arr_longitude'] <= -66) & (24 <= flights['arr_latitude']) & (flights['arr_latitude'] <= 50)
            flights = flights[in_window_arr]
            flights[['dep_mx2t', 'dep_mn2t', 'dep_tp', 'dep_mxtpr', 'dep_fg10', 'dep_u10', 'dep_v10']] = flights.apply(
                lambda x: weather_data(x['Year'], x['Month'], x['DayofMonth'], x['CRSDepTime'], x['dep_longitude'], x['dep_latitude'], alt_weather_1, alt_weather_2), 
                axis = 1, result_type = 'expand')
            flights[['arr_mx2t', 'arr_mn2t', 'arr_tp', 'arr_mxtpr', 'arr_fg10', 'arr_u10', 'arr_v10']] = flights.apply(
                lambda x: weather_data(x['Year'], x['Month'], x['DayofMonth'], x['CRSArrTime'], x['arr_longitude'], x['arr_latitude'], alt_weather_1, alt_weather_2), 
                axis = 1, result_type = 'expand')
            
            #Add in the time lag
            flights[['dep_mx2t_2', 'dep_mn2t_2', 'dep_tp_2', 'dep_mxtpr_2', 'dep_fg10_2', 'dep_u10_2', 'dep_v10_2']] = flights.apply(
                lambda x: weather_data_2(x['Year'], x['Month'], x['DayofMonth'], x['CRSDepTime'], x['dep_longitude'], x['dep_latitude'], alt_weather_1, alt_weather_2), 
                axis = 1, result_type = 'expand')
            flights[['arr_mx2t_2', 'arr_mn2t_2', 'arr_tp_2', 'arr_mxtpr_2', 'arr_fg10_2', 'arr_u10_2', 'arr_v10_2']] = flights.apply(
                lambda x: weather_data_2(x['Year'], x['Month'], x['DayofMonth'], x['CRSArrTime'], x['arr_longitude'], x['arr_latitude'], alt_weather_1, alt_weather_2), 
                axis = 1, result_type = 'expand')
            flights[['dep_mx2t_4', 'dep_mn2t_4', 'dep_tp_4', 'dep_mxtpr_4', 'dep_fg10_4', 'dep_u10_4', 'dep_v10_4']] = flights.apply(
                lambda x: weather_data_2(x['Year'], x['Month'], x['DayofMonth'], x['CRSDepTime'], x['dep_longitude'], x['dep_latitude'], alt_weather_1, alt_weather_2), 
                axis = 1, result_type = 'expand')
            flights[['arr_mx2t_4', 'arr_mn2t_4', 'arr_tp_4', 'arr_mxtpr_4', 'arr_fg10_4', 'arr_u10_4', 'arr_v10_4']] = flights.apply(
                lambda x: weather_data_2(x['Year'], x['Month'], x['DayofMonth'], x['CRSArrTime'], x['arr_longitude'], x['arr_latitude'], alt_weather_1, alt_weather_2), 
                axis = 1, result_type = 'expand')
            


            for column in flights.columns:
                if flights[column].dtype == 'int64':
                    flights[column] = flights[column].astype('int32')
                if flights[column].dtype == 'float64':
                    flights[column] = flights[column].astype('float32')
            flights.to_csv(path_dump + f'{year}_{month}_with_weather.csv')
        

In [13]:
## Only add time lag for 2 and not 4.

for year in years:

    # Conditions are based on the months of each year we are using, each month is then treated in the same way
    if year == 2020:

        # Opens flight and weather files, adds departure and arrival longitute and latitude data to flights
        # Excludes any leftover rows missing information from previous pre-processing and any flights whose origin or destination is not in the continental USA
        flights = pd.read_csv(path_flights + 'cleaned_2020_12.csv')
        alt_weather_1 = xr.open_dataset(path_grib + 'weather_2020_dec_step.nc')
        alt_weather_2 = xr.open_dataset(path_grib + 'weather_2020_dec_uv.nc')
        flights[['dep_longitude', 'dep_latitude']] = flights.apply(lambda x: lon_lat(x['Origin']), axis = 1, result_type = 'expand')
        flights[['arr_longitude', 'arr_latitude']] = flights.apply(lambda x: lon_lat(x['Dest']), axis = 1, result_type = 'expand')
        flights = flights.dropna()
        in_window_dep = (-125 <= flights['dep_longitude']) & (flights['dep_longitude'] <= -66) & (24 <= flights['dep_latitude']) & (flights['dep_latitude'] <= 50)
        flights = flights[in_window_dep]
        in_window_arr = (-125 <= flights['arr_longitude']) & (flights['arr_longitude'] <= -66) & (24 <= flights['arr_latitude']) & (flights['arr_latitude'] <= 50)
        flights = flights[in_window_arr]

        # Adds weather data to flights file, convert appropriate column types to less memory-intensive options - 
        flights[['dep_mx2t', 'dep_mn2t', 'dep_tp', 'dep_mxtpr', 'dep_fg10', 'dep_u10', 'dep_v10']] = flights.apply(
            lambda x: weather_data(x['Year'], x['Month'], x['DayofMonth'], x['CRSDepTime'], x['dep_longitude'], x['dep_latitude'], alt_weather_1, alt_weather_2), 
            axis = 1, result_type = 'expand')
        flights[['arr_mx2t', 'arr_mn2t', 'arr_tp', 'arr_mxtpr', 'arr_fg10', 'arr_u10', 'arr_v10']] = flights.apply(
            lambda x: weather_data(x['Year'], x['Month'], x['DayofMonth'], x['CRSDepTime'], x['arr_longitude'], x['arr_latitude'], alt_weather_1, alt_weather_2), 
            axis = 1, result_type = 'expand')
        for column in flights.columns:
            if flights[column].dtype == 'int64':
                flights[column] = flights[column].astype('int32')
            if flights[column].dtype == 'float64':
                flights[column] = flights[column].astype('float32')
        flights.to_csv(path_dump + '2020_12_with_weather.csv')
        

    elif year == 2025:
        alt_weather_1 = xr.open_dataset(path_grib + 'weather_2025_jan_to_nov_step.nc')
        alt_weather_2 = xr.open_dataset(path_grib + 'weather_2025_jan_to_nov_uv.nc')
        for month in range(1,12):
            flights = pd.read_csv(path_flights + f'cleaned_2025_{month}.csv')
            flights[['dep_longitude', 'dep_latitude']] = flights.apply(lambda x: lon_lat(x['Origin']), axis = 1, result_type = 'expand')
            flights[['arr_longitude', 'arr_latitude']] = flights.apply(lambda x: lon_lat(x['Dest']), axis = 1, result_type = 'expand')
            flights = flights.dropna()
            in_window_dep = (-125 <= flights['dep_longitude']) & (flights['dep_longitude'] <= -66) & (24 <= flights['dep_latitude']) & (flights['dep_latitude'] <= 50)
            flights = flights[in_window_dep]
            in_window_arr = (-125 <= flights['arr_longitude']) & (flights['arr_longitude'] <= -66) & (24 <= flights['arr_latitude']) & (flights['arr_latitude'] <= 50)
            flights = flights[in_window_arr]
            flights[['dep_mx2t', 'dep_mn2t', 'dep_tp', 'dep_mxtpr', 'dep_fg10', 'dep_u10', 'dep_v10']] = flights.apply(
                lambda x: weather_data(x['Year'], x['Month'], x['DayofMonth'], x['CRSDepTime'], x['dep_longitude'], x['dep_latitude'], alt_weather_1, alt_weather_2), 
                axis = 1, result_type = 'expand')
            flights[['arr_mx2t', 'arr_mn2t', 'arr_tp', 'arr_mxtpr', 'arr_fg10', 'arr_u10', 'arr_v10']] = flights.apply(
                lambda x: weather_data(x['Year'], x['Month'], x['DayofMonth'], x['CRSDepTime'], x['arr_longitude'], x['arr_latitude'], alt_weather_1, alt_weather_2), 
                axis = 1, result_type = 'expand')
            for column in flights.columns:
                if flights[column].dtype == 'int64':
                    flights[column] = flights[column].astype('int32')
                if flights[column].dtype == 'float64':
                    flights[column] = flights[column].astype('float32')
            flights.to_csv(path_dump + f'2025_{month}_with_weather.csv')
                
            

    else: 
        alt_weather_1 = xr.open_dataset(path_grib + f'weather_{year}_jan_to_dec_step.nc')
        alt_weather_2 = xr.open_dataset(path_grib + f'weather_{year}_jan_to_dec_uv.nc')
        for month in range(1,13):
            flights = pd.read_csv(path_flights + f'cleaned_{year}_{month}.csv')
            flights[['dep_longitude', 'dep_latitude']] = flights.apply(lambda x: lon_lat(x['Origin']), axis = 1, result_type = 'expand')
            flights[['arr_longitude', 'arr_latitude']] = flights.apply(lambda x: lon_lat(x['Dest']), axis = 1, result_type = 'expand')
            flights = flights.dropna()
            in_window_dep = (-125 <= flights['dep_longitude']) & (flights['dep_longitude'] <= -66) & (24 <= flights['dep_latitude']) & (flights['dep_latitude'] <= 50)
            flights = flights[in_window_dep]
            in_window_arr = (-125 <= flights['arr_longitude']) & (flights['arr_longitude'] <= -66) & (24 <= flights['arr_latitude']) & (flights['arr_latitude'] <= 50)
            flights = flights[in_window_arr]
            flights[['dep_mx2t', 'dep_mn2t', 'dep_tp', 'dep_mxtpr', 'dep_fg10', 'dep_u10', 'dep_v10']] = flights.apply(
                lambda x: weather_data(x['Year'], x['Month'], x['DayofMonth'], x['CRSDepTime'], x['dep_longitude'], x['dep_latitude'], alt_weather_1, alt_weather_2), 
                axis = 1, result_type = 'expand')
            flights[['arr_mx2t', 'arr_mn2t', 'arr_tp', 'arr_mxtpr', 'arr_fg10', 'arr_u10', 'arr_v10']] = flights.apply(
                lambda x: weather_data(x['Year'], x['Month'], x['DayofMonth'], x['CRSArrTime'], x['arr_longitude'], x['arr_latitude'], alt_weather_1, alt_weather_2), 
                axis = 1, result_type = 'expand')
            
            #Add in the time lag
            flights[['dep_mx2t_2', 'dep_mn2t_2', 'dep_tp_2', 'dep_mxtpr_2', 'dep_fg10_2', 'dep_u10_2', 'dep_v10_2']] = flights.apply(
                lambda x: weather_data_2(x['Year'], x['Month'], x['DayofMonth'], x['CRSDepTime'], x['dep_longitude'], x['dep_latitude'], alt_weather_1, alt_weather_2), 
                axis = 1, result_type = 'expand')
            flights[['arr_mx2t_2', 'arr_mn2t_2', 'arr_tp_2', 'arr_mxtpr_2', 'arr_fg10_2', 'arr_u10_2', 'arr_v10_2']] = flights.apply(
                lambda x: weather_data_2(x['Year'], x['Month'], x['DayofMonth'], x['CRSArrTime'], x['arr_longitude'], x['arr_latitude'], alt_weather_1, alt_weather_2), 
                axis = 1, result_type = 'expand')

            for column in flights.columns:
                if flights[column].dtype == 'int64':
                    flights[column] = flights[column].astype('int32')
                if flights[column].dtype == 'float64':
                    flights[column] = flights[column].astype('float32')
            flights.to_csv(path_dump + f'{year}_{month}_with_weather.csv')
        

NameError: name 'xr' is not defined